## Batch Diabetes Prediction

This cell will:

1. Create `new_patients.csv` automatically if it doesn't exist  
2. Load trained diabetes model  
3. Predict diabetes probability  
4. Save results to CSV  

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

In [3]:
MODEL_DIR = Path("models")
MODEL_PATH = MODEL_DIR / "best_model_overall.joblib"

package = joblib.load(MODEL_PATH)

model = package["model"]
threshold = package["threshold"]
feature_columns = package["feature_columns"]

print("Model loaded successfully")
print("Features used:", feature_columns)

Model loaded successfully
Features used: ['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time']


In [14]:
NEW_PATIENTS_PATH = Path("new_patients.csv")

if (not NEW_PATIENTS_PATH.exists()) or NEW_PATIENTS_PATH.stat().st_size == 0:
    print("Creating new_patients.csv automatically...")

    np.random.seed(42)
    n_samples = 100

    new_patients = pd.read_csv('heart_failure_clinical_records.csv')

    new_patients.to_csv(NEW_PATIENTS_PATH, index=False)
    print("new_patients.csv created.")

else:
    print("Using existing new_patients.csv")
    new_patients = pd.read_csv(NEW_PATIENTS_PATH)

print("Shape:", new_patients.shape)
display(new_patients.head())


Using existing new_patients.csv
Shape: (100, 11)


,age,anaemia,creatinine_phosphokinase,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time
0,58,1,704,55,0,285158.780403,1.510505,120,0,1,42
1,71,0,220,38,0,309357.475375,2.152915,140,1,1,101
2,48,1,590,45,1,331522.833431,1.140099,139,0,1,6
3,34,0,85,22,0,393963.019866,2.291046,132,0,1,180
4,62,1,574,38,1,279075.087075,1.278403,147,0,0,282
...,...,...,...,...,...,...,...,...,...,...,...
95,33,1,236,50,1,257248.506844,0.655469,124,0,1,288
96,50,0,375,54,0,337717.766948,2.448790,149,1,1,179
97,67,0,513,52,1,338635.718521,2.472421,149,0,0,125
98,34,0,398,40,1,175780.967209,1.896323,146,0,1,23


In [15]:
# ---------------------------------------
# Fix missing columns automatically
# ---------------------------------------
missing_cols = [col for col in feature_columns if col not in new_patients.columns]

if len(missing_cols) > 0:
    print("Adding missing columns:", missing_cols)

    for col in missing_cols:
        # smart defaults
        if col in ["anaemia", "diabetes", "high_blood_pressure", "sex", "smoking"]:
            new_patients[col] = 0
        else:
            new_patients[col] = new_patients.select_dtypes(include=[np.number]).mean().mean()

# ensure correct column order
X_new = new_patients[feature_columns]

# ---------------------------------------
# Predict
# ---------------------------------------
probabilities = model.predict_proba(X_new)[:, 1]
predictions = (probabilities >= threshold).astype(int)

new_patients["diabetes_probability"] = probabilities
new_patients["diabetes_prediction"] = predictions

display(new_patients.head())

Adding missing columns: ['diabetes']


,age,anaemia,creatinine_phosphokinase,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,diabetes,diabetes_probability,diabetes_prediction
0,58,1,704,55,0,285158.780403,1.510505,120,0,1,42,0,0.820459,1
1,71,0,220,38,0,309357.475375,2.152915,140,1,1,101,0,0.372894,0
2,48,1,590,45,1,331522.833431,1.140099,139,0,1,6,0,0.807670,1
3,34,0,85,22,0,393963.019866,2.291046,132,0,1,180,0,0.578057,1
4,62,1,574,38,1,279075.087075,1.278403,147,0,0,282,0,0.084569,0


In [16]:
new_patients.drop(columns = ['diabetes']).head()

,age,anaemia,creatinine_phosphokinase,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,diabetes_probability,diabetes_prediction
0,58,1,704,55,0,285158.780403,1.510505,120,0,1,42,0.820459,1
1,71,0,220,38,0,309357.475375,2.152915,140,1,1,101,0.372894,0
2,48,1,590,45,1,331522.833431,1.140099,139,0,1,6,0.807670,1
3,34,0,85,22,0,393963.019866,2.291046,132,0,1,180,0.578057,1
4,62,1,574,38,1,279075.087075,1.278403,147,0,0,282,0.084569,0


In [19]:
df

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,55.0,0,748,0,45,0,263358.03,1.3,137,1,1,88,0
1,65.0,0,56,0,25,0,305000.00,5.0,130,1,0,207,0
2,45.0,0,582,1,38,0,319000.00,0.9,140,0,0,244,0
3,60.0,1,754,1,40,1,328000.00,1.2,126,1,0,90,0
4,95.0,1,582,0,30,0,461000.00,2.0,132,1,0,50,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,45.0,0,582,1,55,0,543000.00,1.0,132,0,0,250,0
4996,60.0,1,582,0,30,1,127000.00,0.9,145,0,0,95,0
4997,95.0,1,112,0,40,1,196000.00,1.0,138,0,0,24,1
4998,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [20]:
new_patients

,age,anaemia,creatinine_phosphokinase,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,diabetes,diabetes_probability,diabetes_prediction
0,58,1,704,55,0,285158.780403,1.510505,120,0,1,42,0,0.820459,1
1,71,0,220,38,0,309357.475375,2.152915,140,1,1,101,0,0.372894,0
2,48,1,590,45,1,331522.833431,1.140099,139,0,1,6,0,0.807670,1
3,34,0,85,22,0,393963.019866,2.291046,132,0,1,180,0,0.578057,1
4,62,1,574,38,1,279075.087075,1.278403,147,0,0,282,0,0.084569,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,33,1,236,50,1,257248.506844,0.655469,124,0,1,288,0,0.354541,0
96,50,0,375,54,0,337717.766948,2.448790,149,1,1,179,0,0.340179,0
97,67,0,513,52,1,338635.718521,2.472421,149,0,0,125,0,0.345953,0
98,34,0,398,40,1,175780.967209,1.896323,146,0,1,23,0,0.748662,1
